# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking** (learning-to-rank / scoring).

Refresh Prediction for this capstone is not “predict decline yes/no.” Editors have finite weekly capacity; the business decision is **which pages to refresh first**, not whether a single URL is deteriorating. That decision needs an **ordered** output—a priority score per page and a queue sorted by expected traffic recovery if refreshed this week. A classifier can estimate components of risk, but the deliverable is a **rank**: the top *K* URLs an editor should touch before anything else. Classification answers a different question (membership in a decline bucket) and does not, by itself, allocate scarce hours across hundreds of in-scope pages per client.

## 2. Target or proxy

**Ideal target (not in the starter CSV):** measured **traffic recovered after a refresh** in a forward window—true recoverable impressions we would only see post-intervention.

**Proxy (observable now):** an **expected-recovery score** whose units are *recoverable impressions at stake this week*, built only from trailing-window fields we can inspect today. Intuitively: pages that look like they are **losing demand**, carry **enough impressions to matter**, and sit in a **position tier where CTR upside is still on the table** should score above pages that fail any leg of that tripod.

Plain English: *expected recovery rises when a page shows decline-like movement in recent impressions, when `impressions_90d` is large enough that a refresh could move meaningful clicks, and when `position_tier` / `ctr` imply headroom versus the SERP CTR cliff we measured in w01 (e.g. materially higher mean `ctr` in `top_3` than in `striking`).*

The raw file does **not** ship `is_declining_label`; the nearest observable decline flag is **`trend_direction`** (and its companion **`trend_pct`**), which the data dictionary defines from last-30d vs prev-30d impressions. We treat “in decline” for this proxy as **`trend_direction == "down"`**—a snapshot label for framing, not a claim of causal recovery.

```
expected_recovery_proxy
  ∝  decline_indicator(trend_direction)   # e.g. 1 when "down", else down-weighted
   × impressions_90d                      # exposure at stake
   × ctr_upside(position_tier, ctr)       # non-linear payoff from tier / current CTR
```
(Do not implement here—this notebook only fixes the contract.)

## 3. Success metric

**Primary:** **Precision@K** with **K = 50**, matching the starter pipeline’s headline comparison—of the 50 pages we rank highest for refresh this week, what fraction are truly high-priority under the held-out label (`is_declining_label` in the processed feature path / equivalent decline definition used in `scripts/03_train_model.py`).

**Secondary:** **NDCG@50**, rank-aware and recovery-weighted in spirit: it penalizes putting a high-exposure declining page at rank 40 while a low-stakes page sits at rank 5, even if both are technically “declining.”

**What “good” means:** beat the deterministic hand-written baseline from `scripts/02_baseline_score.py` on **Precision@50**, and report that gap under **client-holdout** validation (`scripts/03_train_model.py`—no client’s pages in both train and test). In-sample leaderboard numbers do not count; the honest claim is generalization to held-out clients, not memorizing one client’s quirks.